# Pipeline Binário (50% vs 75%) — RandomForest + Regressão Logística + XGBoost, com CV Aninhada

**O que mudou em relação ao notebook original (`Pipeline_RF_LogReg_XGB.ipynb`):**

1. **Classificação binária, só 50% vs 75%** — os arquivos de condição 25% são removidos
   na etapa de limpeza (Seção 2). Os três modelos do notebook original continuam:
   **RandomForest**, **Regressão Logística** e **XGBoost**.
2. **CV aninhada (nested Leave-One-Rat-Out)** — no notebook original, a escolha de
   `passo_pct` + hiperparâmetros e a avaliação final usavam os **mesmos folds** de
   Leave-One-Rat-Out, o que infla um pouco a acurácia reportada. Aqui, para cada rato
   deixado de fora na avaliação final (**loop externo**), a escolha de configuração
   (`passo_pct`, com/sem seleção de features, hiperparâmetros) é feita usando **só os
   ratos de treino** desse fold, com outro Leave-One-Rat-Out **dentro** deles (**loop
   interno**). O rato de teste nunca influencia a escolha de configuração usada para
   avaliá-lo — inclusive a remoção de atributos correlacionados agora também é decidida
   só com dados de treino (ver `montar_dataset_treino_teste`).
3. **Grades de hiperparâmetros** — ampliadas nesta versão (mais valores de `passo_pct`
   e mais combinações de hiperparâmetros por modelo) para dar mais chance de achar uma
   configuração melhor. Isso deixa a execução mais lenta; os comentários no código
   indicam onde cortar de volta se ficar inviável no seu computador.
4. **`pasta_multiclasse` corrigido** — no notebook original essa variável nunca era
   definida (dava `NameError` ao tentar salvar). Aqui já vem definida na Seção 1.
5. **Teste de permutação simplificado** — refazer a seleção de configuração para cada
   permutação multiplicaria o custo de novo (por ~50-100x). Aqui o teste reaproveita a
   configuração já escolhida em cada fold externo (a mesma do resultado real) e só
   re-treina/reavalia com os rótulos embaralhados por arquivo. Isso é uma simplificação
   prática, documentada no código — deixe isso explícito se a professora perguntar.
6. **Seção 9 (comparação final + exportação dos resultados)** foi escrita — no notebook
   original essa seção só existia no sumário, não no código.
7. **Warning do sklearn (`sklearn.utils.parallel.delayed`...)** corrigido — o
   `RandomForestClassifier` usado por padrão *dentro do seletor de features*
   (`montar_pipeline_binaria`) ainda estava com `n_jobs=-1`; como ele roda dentro do
   `GridSearchCV` (que também usa `n_jobs=-1`), isso causava paralelismo aninhado, e
   é essa combinação específica que dispara o warning. Agora está `n_jobs=1`.

> **Sobre a meta de ~70% de acurácia:** ampliei as grades de busca nesta versão para dar
> uma chance real de melhorar (isso é legítimo, continua dentro da CV aninhada). Mas vale
> alinhar a expectativa: com só 7 ratos, cada fold de teste tem 13-32 amostras, e a
> variação entre ratos é grande (nos seus resultados, folds individuais já vão de ~0.23
> a ~0.81 de acurácia balanceada) — então a média de 7 números tão variáveis tem uma
> margem de erro grande. Pode ser que 70% simplesmente não seja o valor "verdadeiro" para
> esse conjunto de dados com este conjunto de atributos, e forçar isso via busca mais
> agressiva de hiperparâmetros correria o risco de reintroduzir o mesmo problema que a
> CV aninhada foi feita pra evitar (ver a conversa anterior sobre a CV inflando a
> acurácia). Se depois de ampliar a grade o número não subir muito, o resultado honesto
> (ex.: ~60-65%, significativo pelo teste de permutação) é um resultado legítimo pra
> reportar -- melhor que perseguir um número redondo à custa de rigor metodológico.

> ⚠️ Este notebook **não foi executado aqui** — não tenho acesso aos seus arquivos
> `.int` / `features_all.csv`. Rode do zero, célula por célula, no seu ambiente.
> Como a CV aninhada é bem mais cara, considere rodar em background / durante a noite
> se o dataset for grande.


## 1. Configuração e Carregamento dos Dados

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.ensemble import RandomForestClassifier  # usado só como selector interno do SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

try:
    import shap
    SHAP_DISPONIVEL = True
except ImportError:
    SHAP_DISPONIVEL = False
    print("Pacote 'shap' não encontrado. Instale com: pip install shap")

try:
    from xgboost import XGBClassifier
    XGBOOST_DISPONIVEL = True
except ImportError:
    XGBOOST_DISPONIVEL = False
    print("Pacote 'xgboost' não encontrado. Instale com: pip install xgboost")

warnings.filterwarnings("ignore")
# Filtro extra (cinto e suspensório) para o warning de paralelismo aninhado do
# sklearn -- a causa real já foi corrigida (estimadores internos com n_jobs=1
# em vez de -1, evitando nested parallelism), isso aqui é só reforço.
warnings.filterwarnings("ignore", message=".*should be used with.*Parallel.*")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

TOP_N_FEATURES_GRID = [5, 10, 15, 20, 25]  # ampliado (mais opções de nº de atributos selecionados)

# Caminho do features_all
CAMINHOS_CANDIDATOS = [
    Path("databases/features_all.csv"),
]

# Limite mínimo de duração do vídeo (segundos)
DURACAO_MINIMA_S = 30

FEATURE_BOOLEANA = "pac_sl_alerta"
FEATURES_POR_AREA = [
    "mean", "std", "rms", "kurtosis", "skewness", "pico_a_pico", "pct_outlier_3s",
    "pot_delta", "pot_theta", "pot_beta", "pot_gamma_lento", "pot_gamma_rapido",
    "theta_gamma_lento_ratio", "theta_gamma_rapido_ratio", "delta_theta_ratio",
    "entropia_espectral", "centroide_hz",
    "pac_theta-gamma_rapido", "pac_theta-gamma_lento", "pac_sl_alerta",
]

# Grade de "pct" (tamanho da janela percentual) testada na escolha do melhor.
# IMPORTANTE: só usamos valores que são divisores exatos de 100. A função
# montar_dataset_modelagem faz n_bins = round(100 / passo_pct); com valores que
# não dividem 100 exatamente isso gera bins tortos (ex.: passo_pct=30 ou 35
# viravam n_bins=3, ou seja, janelas de 33% em vez de 30%/35%) e também
# duplicava configurações (ex.: passo_pct=40 e 45 caíam ambos em n_bins=2,
# igual ao passo_pct=50 -- rodando o mesmo modelo 3x à toa). Com divisores
# exatos de 100 cada passo_pct vira um n_bins único e as janelas batem
# exatamente com o rótulo (5% é 5% mesmo, não 4.76%).
PASSO_PCT_GRID = [4, 5, 10, 20, 25, 50]
print(f"Candidatos de passo_pct a testar: {PASSO_PCT_GRID}")
print(f"(n_bins correspondentes: {[round(100 / p) for p in PASSO_PCT_GRID]} -- "
      f"todos exatos, sem arredondamento)")

# Classificação binária: só 50% vs 75% (25% é removido na Seção 2)
CONDICOES_ALVO = [50, 75]

caminho_dados = next((p for p in CAMINHOS_CANDIDATOS if p.exists()), None)
if caminho_dados is None:
    raise FileNotFoundError(
        "features_all não encontrado. Ajuste CAMINHOS_CANDIDATOS para apontar "
        "para o seu features_all.parquet ou features_all.csv."
    )

if caminho_dados.suffix == ".parquet":
    df_bruto = pd.read_parquet(caminho_dados)
else:
    df_bruto = pd.read_csv(caminho_dados)

print(f" Carregado: {caminho_dados}")
print(f"   Shape: {df_bruto.shape[0]:,} linhas × {df_bruto.shape[1]} colunas")

# Pasta de saída -- CORRIGIDO: no notebook original esta variável nunca era
# definida, e o notebook dava NameError ao tentar salvar os resultados.
pasta_multiclasse = Path("resultados_binario_50_75")
pasta_multiclasse.mkdir(exist_ok=True)
print(f"   Resultados serão salvos em: {pasta_multiclasse.resolve()}")


## 2. Limpeza dos Dados
Remove gravações do grupo NOR, arquivos com duração menor que o mínimo estipulado,
**e as gravações de condição 25%** (a análise agora é binária: 50% vs 75%).

In [ ]:
duracao_por_stem = df_bruto.groupby("stem")["segundo"].max()
stems_curtos = duracao_por_stem[duracao_por_stem < DURACAO_MINIMA_S]

n_arquivos_antes = df_bruto["stem"].nunique()

stems_nor = df_bruto.loc[
    df_bruto["stem"].str.contains("nor", case=False, na=False),
    "stem"
].unique()

# NOVO: remove também os arquivos de condição 25% -- este notebook classifica
# só 50% vs 75%.
condicao_numerica = pd.to_numeric(df_bruto["condicao"], errors="coerce")
stems_25pct = df_bruto.loc[np.isclose(condicao_numerica, 0.25, atol=1e-6), "stem"].unique()

df_limpo = df_bruto[~df_bruto["stem"].isin(stems_nor)].copy()
df_limpo = df_limpo[~df_limpo["stem"].isin(stems_curtos.index)].copy()
df_limpo = df_limpo[~df_limpo["stem"].isin(stems_25pct)].copy()

n_arquivos_depois = df_limpo["stem"].nunique()

print("=" * 60)
print(f"Arquivos antes das exclusões : {n_arquivos_antes}")
print(f"Arquivos NOR removidos       : {len(stems_nor)}")
print(f"Arquivos < {DURACAO_MINIMA_S}s removidos : {len(stems_curtos)}")
print(f"Arquivos de 25% removidos    : {len(stems_25pct)}")
print(f"Arquivos após as exclusões   : {n_arquivos_depois}")
print("=" * 60)

AREAS = sorted({c[:-len("_mean")] for c in df_limpo.columns if c.endswith("_mean")})
print(f"\nÁreas anatômicas encontradas ({len(AREAS)}): {AREAS}")

condicoes_restantes = sorted(pd.to_numeric(df_limpo["condicao"], errors="coerce").dropna().unique())
print(f"Condições restantes no dataset: {condicoes_restantes}  (deve conter só 0.5 e 0.75)")
assert set(np.round(condicoes_restantes, 2)) <= {0.5, 0.75}, (
    "Ainda há condições fora de {50%, 75%} em df_limpo -- confira a coluna 'condicao' "
    "(ex.: pode haver algum valor gravado com formatação diferente de '0.25')."
)


## 3. Engenharia de Atributos (parametrizada por `n_bins`)

Mesma lógica do notebook original (`transformar_wide_por_arquivo_area` +
`remover_colunas_correlacionadas`), **mais uma função nova**:
`montar_dataset_treino_teste`, usada pela CV aninhada para montar treino e teste
separadamente, garantindo que:

- as colunas de área (dummies) fiquem **fixas** (via categorias fixas = `AREAS`),
  mesmo que o rato de teste não tenha alguma área anatômica registrada;
- a remoção de atributos correlacionados seja decidida **olhando só para o treino**
  — o rato de teste nunca influencia quais colunas sobrevivem.

In [ ]:
def atribuir_faixa_percentual(n_segundos, n_bins):
    posicao_relativa = (np.arange(n_segundos) + 1) / n_segundos
    faixa = np.ceil(posicao_relativa * n_bins).astype(int)
    return np.clip(faixa, 1, n_bins)


def transformar_wide_por_arquivo_area(df_long, areas, features_por_area, n_bins):

    passo_pct = 100 // n_bins
    rotulos_pct = [passo_pct * b for b in range(1, n_bins + 1)]

    df_long = df_long.sort_values(["stem", "segundo"]).copy()

    linhas_resultado = []

    for stem, grupo in df_long.groupby("stem", sort=False):

        grupo = grupo.sort_values("segundo")
        grupo = grupo.assign(_faixa_pct=atribuir_faixa_percentual(len(grupo), n_bins))

        rato = grupo["rato"].iloc[0]
        trial = grupo["trial"].iloc[0]
        condicao = grupo["condicao"].iloc[0]

        for area in areas:

            col_ref = f"{area}_mean"
            area_registrada = col_ref in grupo.columns and grupo[col_ref].notna().any()
            if not area_registrada:
                continue

            linha = {
                "arquivo_area": f"{rato}_{trial}_{area}",
                "stem": stem, "rato": rato, "trial": trial,
                "condicao": condicao, "area": area, "duracao_s": len(grupo),
            }

            ESTATISTICAS_POR_JANELA = ["mean", "std", "min", "max", "median"]

            for feature in features_por_area:
                if feature == FEATURE_BOOLEANA:
                    continue

                coluna = f"{area}_{feature}"
                if coluna not in grupo.columns:
                    for pct in rotulos_pct:
                        for estat in ESTATISTICAS_POR_JANELA:
                            linha[f"{feature}_{estat}_{pct}pct"] = np.nan
                    continue

                for faixa, pct in zip(range(1, n_bins + 1), rotulos_pct):
                    valores = grupo.loc[grupo["_faixa_pct"] == faixa, coluna]
                    linha[f"{feature}_mean_{pct}pct"] = valores.mean()
                    linha[f"{feature}_std_{pct}pct"] = (
                        valores.std() if len(valores) > 1 else np.nan
                    )
                    linha[f"{feature}_min_{pct}pct"] = (
                        valores.min() if len(valores) else np.nan
                    )
                    linha[f"{feature}_max_{pct}pct"] = (
                        valores.max() if len(valores) else np.nan
                    )
                    linha[f"{feature}_median_{pct}pct"] = valores.median()

            coluna_bool = f"{area}_{FEATURE_BOOLEANA}"
            for faixa, pct in zip(range(1, n_bins + 1), rotulos_pct):
                if coluna_bool in grupo.columns:
                    valores = grupo.loc[grupo["_faixa_pct"] == faixa, coluna_bool]
                    linha[f"{FEATURE_BOOLEANA}_true_rate_{pct}pct"] = (
                        (valores == True).mean() if len(valores) else np.nan
                    )
                else:
                    linha[f"{FEATURE_BOOLEANA}_true_rate_{pct}pct"] = np.nan

            linhas_resultado.append(linha)

    return pd.DataFrame(linhas_resultado)


def remover_colunas_correlacionadas(X_ref, colunas_protegidas, limiar=0.8):
    """Identifica atributos com |correlação| > limiar (colunas protegidas ficam de fora)."""
    colunas_avaliar = [c for c in X_ref.columns if c not in colunas_protegidas]
    corr = X_ref[colunas_avaliar].corr().abs()
    mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
    upper = corr.where(mask)
    return [c for c in upper.columns if any(upper[c] > limiar)]


def _montar_wide_com_derivadas(df_long, n_bins):
    """Wide transform + colunas derivadas (_media_geral, taxa geral do PAC alerta)
    + dummy de área com categorias FIXAS (= AREAS), pra manter as mesmas colunas
    em treino e teste mesmo que algum rato não tenha determinada área."""
    df_pct = transformar_wide_por_arquivo_area(df_long, AREAS, FEATURES_POR_AREA, n_bins=n_bins)

    features_continuas = [f for f in FEATURES_POR_AREA if f != FEATURE_BOOLEANA]
    passo_pct_real = 100 // n_bins
    pcts = [passo_pct_real * b for b in range(1, n_bins + 1)]

    for feature in features_continuas:
        cols = [f"{feature}_mean_{p}pct" for p in pcts]
        df_pct[f"{feature}_media_geral"] = df_pct[cols].mean(axis=1)

    df_pct["pac_sl_alerta_taxa_geral"] = df_pct[
        [f"pac_sl_alerta_true_rate_{p}pct" for p in pcts]
    ].mean(axis=1)

    df_pct["condicao"] = pd.to_numeric(df_pct["condicao"])
    df_pct = df_pct.drop(columns=["trial", "duracao_s", "stem", "arquivo_area"])

    # categorias fixas -> mesmas colunas dummy em treino e teste
    df_pct["area"] = pd.Categorical(df_pct["area"], categories=AREAS)
    df_pct = pd.get_dummies(df_pct, columns=["area"], prefix="area", dtype=int)
    return df_pct


def montar_dataset_modelagem(df_long_limpo, passo_pct, verbose=True):
    """Reproduz a engenharia de atributos completa (transformação wide +
    remoção de correlacionados) para um dado `passo_pct`, usando TODOS os dados
    recebidos. Usada para o "modelo final" de interpretação (SHAP) -- NÃO usada
    para reportar acurácia (isso é feito por `montar_dataset_treino_teste`,
    abaixo, dentro da CV aninhada)."""
    n_bins = max(1, round(100 / passo_pct))
    df_pct = _montar_wide_com_derivadas(df_long_limpo, n_bins)

    groups = df_pct["rato"]
    y = df_pct["condicao"]
    X = df_pct.drop(columns=["rato", "condicao"], errors="ignore")

    colunas_protegidas = [c for c in X.columns if c.startswith("area_")]
    colunas_removidas = remover_colunas_correlacionadas(X, colunas_protegidas)
    features_finais = [c for c in X.columns if c not in colunas_removidas]
    X_final = X[features_finais]

    if verbose:
        print(f"[passo_pct={passo_pct} -> n_bins={n_bins}] "
              f"atributos: {X.shape[1]} -> {X_final.shape[1]} "
              f"({len(colunas_removidas)} removidos por correlação)")

    return X_final, y, groups, colunas_removidas, n_bins


def montar_dataset_treino_teste(df_treino_long, df_teste_long, passo_pct):
    """Monta treino e teste SEPARADAMENTE para um fold da CV aninhada.

    A remoção de atributos correlacionados é decidida usando só `df_treino_long`
    -- o rato de teste (em `df_teste_long`) nunca participa dessa decisão.
    """
    n_bins = max(1, round(100 / passo_pct))

    df_pct_treino = _montar_wide_com_derivadas(df_treino_long, n_bins)
    df_pct_teste = _montar_wide_com_derivadas(df_teste_long, n_bins)

    groups_treino = df_pct_treino["rato"]
    y_treino = df_pct_treino["condicao"]
    X_treino = df_pct_treino.drop(columns=["rato", "condicao"], errors="ignore")

    groups_teste = df_pct_teste["rato"]
    y_teste = df_pct_teste["condicao"]
    X_teste = df_pct_teste.drop(columns=["rato", "condicao"], errors="ignore")

    # remoção de correlacionados decidida SÓ no treino
    colunas_protegidas = [c for c in X_treino.columns if c.startswith("area_")]
    colunas_removidas = remover_colunas_correlacionadas(X_treino, colunas_protegidas)
    features_finais = [c for c in X_treino.columns if c not in colunas_removidas]

    X_treino_final = X_treino[features_finais]
    X_teste_final = X_teste[features_finais]

    return X_treino_final, y_treino, groups_treino, X_teste_final, y_teste, groups_teste, n_bins


print("Funções de engenharia de atributos definidas (com suporte a CV aninhada).")


## 4. Funções Genéricas — Pipeline + CV Aninhada

Diferença principal em relação ao original: em vez de uma função só que escolhe
configuração e avalia nos mesmos folds, agora são duas etapas separadas por fold:

- **`escolher_configuracao_interna`**: roda uma CV interna (LOGO só sobre os ratos
  de treino) pra escolher `passo_pct` + com/sem seleção de features + hiperparâmetros.
- **`treinar_avaliar_aninhado`**: loop externo (1 rato de teste por vez) que chama a
  função acima só com os dados de treino daquele fold, treina com a configuração
  escolhida, e avalia no rato de teste (nunca visto pela escolha de configuração).

In [ ]:
def montar_pipeline_binaria(estimator, usar_selecao_features=True, selector_estimator=None):
    """Pipeline genérico: StandardScaler -> SelectFromModel -> estimator."""
    passos = [("scaler", StandardScaler())]
    if usar_selecao_features:
        sel_est = selector_estimator or RandomForestClassifier(
            n_estimators=300, max_depth=5, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1,  # single-thread: evita paralelismo aninhado
        )
        passos.append(("selector", SelectFromModel(sel_est, threshold=-np.inf, max_features=20)))
    passos.append(("clf", estimator))
    return Pipeline(passos)


def _scorer_balanced_accuracy_silencioso(estimator, X, y):
    y_pred = estimator.predict(X)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        return balanced_accuracy_score(y, y_pred)


def escolher_configuracao_interna(df_treino_long, estimator, param_grid_semsel, param_grid_comsel,
                                   passo_pct_grid, y_transformer=None, rotulo="modelo"):
    """CV INTERNA: escolhe passo_pct, com/sem seleção de features e hiperparâmetros
    usando SÓ os ratos de treino deste fold externo (LOGO interno sobre eles).
    O rato de teste do fold externo nunca aparece aqui -- por isso a escolha de
    configuração não "vaza" informação do rato que será avaliado."""
    y_transformer = y_transformer or (lambda y: y)
    logo_interno = LeaveOneGroupOut()

    melhor_global = None  # (score, passo_pct, usa_selecao, params)

    for passo_pct in passo_pct_grid:
        X_pct, y_pct, groups_pct, _, n_bins = montar_dataset_modelagem(df_treino_long, passo_pct, verbose=False)
        y_pct = y_transformer((y_pct * 100).round().astype(int))

        if groups_pct.nunique() < 2:
            print(f"    [{rotulo}] passo_pct={passo_pct}: menos de 2 ratos de treino -- pulando.")
            continue

        cv_splits_interno = list(logo_interno.split(X_pct, y_pct, groups=groups_pct))

        pipe_semsel = montar_pipeline_binaria(clone(estimator), usar_selecao_features=False)
        busca_semsel = GridSearchCV(
            pipe_semsel, param_grid=param_grid_semsel, scoring=_scorer_balanced_accuracy_silencioso,
            cv=cv_splits_interno, n_jobs=-1, refit=False,
        )
        busca_semsel.fit(X_pct, y_pct)
        candidatos = [(busca_semsel.best_score_, passo_pct, False, dict(busca_semsel.best_params_))]

        pipe_comsel = montar_pipeline_binaria(clone(estimator), usar_selecao_features=True)
        busca_comsel = GridSearchCV(
            pipe_comsel, param_grid=param_grid_comsel, scoring=_scorer_balanced_accuracy_silencioso,
            cv=cv_splits_interno, n_jobs=-1, refit=False,
        )
        busca_comsel.fit(X_pct, y_pct)
        candidatos.append((busca_comsel.best_score_, passo_pct, True, dict(busca_comsel.best_params_)))

        for cand in candidatos:
            if melhor_global is None or cand[0] > melhor_global[0]:
                melhor_global = cand

    score, passo_pct, usa_selecao, params = melhor_global
    print(f"    [{rotulo}] config interna escolhida: passo_pct={passo_pct} | "
          f"seleção={'com' if usa_selecao else 'sem'} | CV interna={score:.3f} | params={params}")
    return {"passo_pct": passo_pct, "usa_selecao": usa_selecao, "params": params}


def treinar_avaliar_aninhado(df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
                              passo_pct_grid=None, y_transformer=None, inverse_label_fn=None):
    """Avaliação com CV ANINHADA (nested Leave-One-Rat-Out):
    - loop EXTERNO: cada rato vira teste uma vez;
    - dentro de cada fold externo, a escolha de passo_pct / seleção de features /
      hiperparâmetros usa só os ratos de TREINO daquele fold (CV interna) --
      o rato de teste nunca participa dessa escolha."""
    passo_pct_grid = passo_pct_grid or PASSO_PCT_GRID
    y_transformer = y_transformer or (lambda y: y)
    ratos = sorted(df_long_limpo["rato"].unique())
    labels_multi = sorted(y_transformer(pd.Series(CONDICOES_ALVO)).tolist())

    resultados_fold = []
    configs_fold = []
    y_true_all, y_pred_all = [], []

    for i, rato_teste in enumerate(ratos, start=1):
        print(f"\n{'='*70}\n[{nome_modelo}] Fold externo {i}/{len(ratos)} -- rato de teste: {rato_teste}\n{'='*70}")

        df_treino_long = df_long_limpo[df_long_limpo["rato"] != rato_teste]
        df_teste_long = df_long_limpo[df_long_limpo["rato"] == rato_teste]

        config = escolher_configuracao_interna(
            df_treino_long, estimator, param_grid_semsel, param_grid_comsel,
            passo_pct_grid, y_transformer=y_transformer, rotulo=nome_modelo,
        )
        configs_fold.append({"rato_teste": rato_teste, **config})

        X_tr, y_tr, _, X_te, y_te, _, _ = montar_dataset_treino_teste(
            df_treino_long, df_teste_long, config["passo_pct"]
        )
        y_tr = y_transformer((y_tr * 100).round().astype(int))
        y_te = y_transformer((y_te * 100).round().astype(int))

        pipe = montar_pipeline_binaria(clone(estimator), config["usa_selecao"])
        pipe.set_params(**config["params"])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            bal_acc = balanced_accuracy_score(y_te, y_pred)
        f1_macro = f1_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)
        precision_macro = precision_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)
        recall_macro = recall_score(y_te, y_pred, labels=labels_multi, average="macro", zero_division=0)

        y_true_all.extend(y_te.tolist())
        y_pred_all.extend(list(y_pred))

        resultados_fold.append({
            "fold": i, "rato_teste": rato_teste, "n_amostras_teste": len(y_te),
            "passo_pct": config["passo_pct"], "usa_selecao_features": config["usa_selecao"],
            "balanced_accuracy": bal_acc, "f1_macro": f1_macro,
            "precision_macro": precision_macro, "recall_macro": recall_macro,
        })
        print(f"  [{nome_modelo}] fold {i}/{len(ratos)} -- teste=rato {rato_teste} (n={len(y_te)}) "
              f"| passo_pct={config['passo_pct']} | bal_acc={bal_acc:.3f} | f1_macro={f1_macro:.3f}")

    df_resultados = pd.DataFrame(resultados_fold)
    matriz_confusao = confusion_matrix(y_true_all, y_pred_all, labels=labels_multi)

    labels_exibicao = [f"{c}%" for c in labels_multi]
    if inverse_label_fn is not None:
        labels_exibicao = [f"{c}%" for c in inverse_label_fn(labels_multi)]

    return {
        "nome_modelo": nome_modelo,
        "df_resultados": df_resultados,
        "df_configs_por_fold": pd.DataFrame(configs_fold),
        "matriz_confusao": matriz_confusao,
        "labels": labels_exibicao,
        "labels_multi": labels_multi,
    }


def relatar_resultado_binario(resultado, titulo, chance_nivel):
    df_resultados = resultado["df_resultados"]

    print(f"\n=== Resumo (CV aninhada) -- {titulo} ===")
    print(f"Acurácia balanceada: {df_resultados['balanced_accuracy'].mean():.3f}"
          f" ± {df_resultados['balanced_accuracy'].std():.3f}")
    print(f"F1 macro       : {df_resultados['f1_macro'].mean():.3f}"
          f" ± {df_resultados['f1_macro'].std():.3f}")
    print(f"Precision macro: {df_resultados['precision_macro'].mean():.3f}"
          f" ± {df_resultados['precision_macro'].std():.3f}")
    print(f"Recall macro   : {df_resultados['recall_macro'].mean():.3f}"
          f" ± {df_resultados['recall_macro'].std():.3f}")

    print(f"\nConfiguração escolhida em cada fold externo (é normal variar entre folds "
          f"-- a CV interna é refeita a cada um):")
    display(resultado["df_configs_por_fold"])

    cm = resultado["matriz_confusao"]
    labels_plot = resultado["labels"]

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels_plot))); ax.set_xticklabels(labels_plot)
    ax.set_yticks(range(len(labels_plot))); ax.set_yticklabels(labels_plot)
    ax.set_xlabel("Predito"); ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão (agregada) -- {titulo}")
    limiar = cm.max() / 2 if cm.max() else 1
    for i in range(len(labels_plot)):
        for j in range(len(labels_plot)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > limiar else "black")
    fig.colorbar(im, ax=ax, label="n° de amostras")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(df_resultados["fold"], df_resultados["balanced_accuracy"], marker="o")
    ax.axhline(chance_nivel, color="black", linestyle=":", label=f"chance ({chance_nivel:.2f})")
    ax.set_xticks(df_resultados["fold"])
    ax.set_xticklabels(df_resultados["rato_teste"], rotation=45, ha="right")
    ax.set_xlabel("Rato de teste (fold externo)"); ax.set_ylabel("balanced_accuracy")
    ax.set_title(f"balanced_accuracy por fold externo -- {titulo}")
    ax.legend()
    plt.tight_layout()
    plt.show()


def treinar_modelo_final_interpretacao(df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
                                        passo_pct_grid=None, y_transformer=None, rotulo="modelo"):
    """Treina um modelo final em TODOS os ratos, só para fins de interpretação
    (SHAP / importância de features) -- NÃO é usado para reportar acurácia.
    A escolha de configuração aqui usa CV normal (não aninhada) sobre todos os
    ratos, o que é aceitável porque nenhuma acurácia é reportada a partir dele."""
    passo_pct_grid = passo_pct_grid or PASSO_PCT_GRID
    y_transformer = y_transformer or (lambda y: y)

    config = escolher_configuracao_interna(
        df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
        passo_pct_grid, y_transformer=y_transformer, rotulo=f"{rotulo} [modelo final]",
    )
    X_final, y_final, groups_final, _, n_bins = montar_dataset_modelagem(
        df_long_limpo, config["passo_pct"], verbose=False
    )
    y_final = y_transformer((y_final * 100).round().astype(int))

    pipe_final = montar_pipeline_binaria(clone(estimator), config["usa_selecao"])
    pipe_final.set_params(**config["params"])
    pipe_final.fit(X_final, y_final)

    return {"pipe_final": pipe_final, "X_final": X_final, "y_final": y_final,
            "groups_final": groups_final, "config": config}


def calcular_e_plotar_shap(pipe, X, rotulo, max_amostras=200):
    """Calcula os valores SHAP do classificador final (já treinado em todos os
    dados) e plota as top-15 features por |SHAP| médio."""
    if not SHAP_DISPONIVEL:
        print(f"[{rotulo}] shap não disponível -- pulando etapa de SHAP.")
        return None

    X_proc = X
    for _, passo in pipe.steps[:-1]:
        X_proc = passo.transform(X_proc)
    selector = pipe.named_steps.get("selector")
    feats = X.columns[selector.get_support()] if selector is not None else X.columns
    X_proc = pd.DataFrame(np.asarray(X_proc), columns=feats, index=X.index)

    n_amostra = min(max_amostras, len(X_proc))
    X_amostra = X_proc.sample(n_amostra, random_state=RANDOM_STATE) if len(X_proc) > n_amostra else X_proc

    clf = pipe.named_steps["clf"]
    # Modelos de árvore (RF, XGBoost) usam TreeExplainer explicitamente com
    # feature_perturbation="tree_path_dependent" -- o shap.Explainer genérico
    # pode cair em erro (NotImplementedError: "Categorical split is not yet
    # supported") em versões recentes do XGBoost com tree_method="hist".
    modelos_arvore = (RandomForestClassifier,) + ((XGBClassifier,) if XGBOOST_DISPONIVEL else ())
    if isinstance(clf, modelos_arvore):
        explainer = shap.TreeExplainer(clf, feature_perturbation="tree_path_dependent")
    else:
        explainer = shap.Explainer(clf, X_proc)
    valores_shap = explainer(X_amostra)

    valores = valores_shap.values
    valores_abs_medios = (
        np.abs(valores).mean(axis=(0, 2)) if valores.ndim == 3 else np.abs(valores).mean(axis=0)
    )
    resumo_shap = pd.Series(valores_abs_medios, index=feats).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    top_shap = resumo_shap.head(15).iloc[::-1]
    ax.barh(top_shap.index, top_shap.values, color="#55A868")
    ax.set_xlabel("|SHAP| médio")
    ax.set_title(f"Top 15 features (SHAP) -- {rotulo}")
    plt.tight_layout()
    plt.show()

    return resumo_shap


def pipeline_binario_aninhado(df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
                               passo_pct_grid=None, y_transformer=None, inverse_label_fn=None,
                               calcular_shap_flag=True, shap_max_amostras=200):
    passo_pct_grid = passo_pct_grid or PASSO_PCT_GRID
    chance_nivel = 1 / len(CONDICOES_ALVO)

    print(f"\n{'#'*70}\n# {nome_modelo} -- CV ANINHADA (nested Leave-One-Rat-Out)\n{'#'*70}")
    resultado = treinar_avaliar_aninhado(
        df_long_limpo, nome_modelo, estimator, param_grid_semsel, param_grid_comsel,
        passo_pct_grid=passo_pct_grid, y_transformer=y_transformer, inverse_label_fn=inverse_label_fn,
    )
    relatar_resultado_binario(resultado, titulo=nome_modelo, chance_nivel=chance_nivel)

    print(f"\n{'#'*70}\n# {nome_modelo} -- modelo final (todos os ratos) p/ interpretação (SHAP)\n{'#'*70}")
    modelo_final = treinar_modelo_final_interpretacao(
        df_long_limpo, estimator, param_grid_semsel, param_grid_comsel,
        passo_pct_grid=passo_pct_grid, y_transformer=y_transformer, rotulo=nome_modelo,
    )

    resumo_shap = None
    if calcular_shap_flag:
        resumo_shap = calcular_e_plotar_shap(
            modelo_final["pipe_final"], modelo_final["X_final"], rotulo=nome_modelo, max_amostras=shap_max_amostras,
        )

    return {
        "nome_modelo": nome_modelo,
        "resultado": resultado,
        "modelo_final": modelo_final,
        "resumo_shap": resumo_shap,
    }


print("Funções genéricas (pipeline + CV aninhada) definidas.")


## 5. RandomForest — Pipeline Completo (CV aninhada)

Grade reduzida pelo mesmo motivo das outras seções (custo multiplicado pelo
número de ratos no loop externo). Mantive `class_weight="balanced"` como no
notebook original.

In [ ]:
# Grade mais ampla (a pedido explícito, aceitando rodar mais devagar):
# +1 valor de n_estimators, +1 de max_depth, +1 de min_samples_leaf, entrou
# min_samples_split, max_features ganhou mais frações contínuas, e
# class_weight agora testa também "balanced_subsample" (reamostra o balanço
# de classe em cada árvore do bagging, não só uma vez -- pode ajudar com
# poucos ratos). Combos por passo_pct (sem contar seleção de atributos):
# 4*5*4*3*4*2 = 1920 (SEMSEL) -- e mais TOP_N_FEATURES_GRID (5) de multiplicador
# no COMSEL. Isso é MUITO mais caro que a grade anterior (ordem de ~10-15x);
# se travar na sua máquina, o primeiro corte recomendado é remover
# min_samples_split ou reduzir max_features para 3 valores.
PARAM_GRID_RF_SEMSEL = {
    "clf__n_estimators": [100, 200, 300],          # 3
    "clf__max_depth": [3, 8, 12, None],            # 4
    "clf__min_samples_leaf": [1, 2, 3, 5],         # 4
    "clf__min_samples_split": [2, 5, 10],          # 3
    "clf__max_features": ["sqrt", 0.3, 0.6],       # 3
    "clf__class_weight": ["balanced", "balanced_subsample"],  # 2
}
PARAM_GRID_RF_COMSEL = {
    "selector__max_features": TOP_N_FEATURES_GRID,
    **PARAM_GRID_RF_SEMSEL,
}

rf_base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)

saida_rf = pipeline_binario_aninhado(
    df_limpo, nome_modelo="RandomForest (50% vs 75%)", estimator=rf_base,
    param_grid_semsel=PARAM_GRID_RF_SEMSEL, param_grid_comsel=PARAM_GRID_RF_COMSEL,
)


## 6. Regressão Logística — Pipeline Completo (CV aninhada)

Grade reduzida em relação ao original: com CV aninhada o custo é multiplicado
pelo número de ratos no loop externo, então fixei `solver`, `class_weight` e
`fit_intercept` nos valores que já haviam se mostrado bons no notebook original,
e deixei `C` como principal hiperparâmetro de busca. Amplie se tiver tempo.

In [ ]:
# Grade bem mais ampla (≈924 combinações): regressão logística é barata de
# treinar, então dá para explorar uma faixa maior de regularização sem um
# custo proibitivo. Testamos L1, L2 e ElasticNet (via l1_ratio), sempre com
# o solver "saga", que suporta as três penalidades. Usamos uma LISTA de
# dicionários para evitar combinações inválidas (por exemplo, l1_ratio com
# L1/L2), reduzindo tempo desperdiçado pelo GridSearchCV.

_LOGREG_C_GRID = [
    1e-5, 3e-5,
    1e-4, 3e-4,
    1e-3, 3e-3,
    1e-2, 3e-2,
    1e-1, 3e-1,
    1, 3,
    10, 30,
    100, 300,
    1000, 3000,
    10000, 30000,
    100000, 300000,
]  # 22 valores

_LOGREG_CLASS_WEIGHT_GRID = [None, "balanced"]

_LOGREG_L1_RATIO_GRID = [
    0.05, 0.15, 0.25, 0.35,
    0.50,
    0.65, 0.75, 0.85, 0.95,
]  # 9 valores

PARAM_GRID_LOGREG_SEMSEL = [
    {
        "clf__penalty": ["l1"],
        "clf__solver": ["saga"],
        "clf__C": _LOGREG_C_GRID,
        "clf__class_weight": _LOGREG_CLASS_WEIGHT_GRID,
        "clf__max_iter": [5000],
    },
    {
        "clf__penalty": ["l2"],
        "clf__solver": ["saga"],
        "clf__C": _LOGREG_C_GRID,
        "clf__class_weight": _LOGREG_CLASS_WEIGHT_GRID,
        "clf__max_iter": [5000],
    },
    {
        "clf__penalty": ["elasticnet"],
        "clf__solver": ["saga"],
        "clf__C": _LOGREG_C_GRID,
        "clf__l1_ratio": _LOGREG_L1_RATIO_GRID,
        "clf__class_weight": _LOGREG_CLASS_WEIGHT_GRID,
        "clf__max_iter": [5000],
    },
]

# COMSEL: mesma lista, só acrescentando selector__max_features em cada bloco.
PARAM_GRID_LOGREG_COMSEL = [
    {**bloco, "selector__max_features": TOP_N_FEATURES_GRID}
    for bloco in PARAM_GRID_LOGREG_SEMSEL
]

logreg_base = LogisticRegression(
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

saida_logreg = pipeline_binario_aninhado(
    df_limpo,
    nome_modelo="Regressão Logística (50% vs 75%)",
    estimator=logreg_base,
    param_grid_semsel=PARAM_GRID_LOGREG_SEMSEL,
    param_grid_comsel=PARAM_GRID_LOGREG_COMSEL,
)

## 7. XGBoost — Pipeline Completo (CV aninhada)

Agora binário (`objective="binary:logistic"`), então os rótulos `50/75` são
recodificados para `0/1` via `y_transformer`, e os relatórios finais são
traduzidos de volta via `inverse_label_fn` (mesma lógica do notebook original).

In [ ]:
if XGBOOST_DISPONIVEL:
    MAPA_XGB = {c: i for i, c in enumerate(CONDICOES_ALVO)}   # {50: 0, 75: 1}
    MAPA_XGB_INV = {i: c for c, i in MAPA_XGB.items()}

    def codificar_y_xgb(y_series):
        return y_series.map(MAPA_XGB)

    def decodificar_y_xgb(labels_codificados):
        return [MAPA_XGB_INV[l] for l in labels_codificados]

    # Grade mais ampla (a pedido explícito): +1 valor de max_depth e
    # learning_rate, subsample/colsample ganharam um ponto intermediário, e
    # entraram reg_lambda (regularização L2 nas folhas) e gamma (ganho mínimo
    # pra abrir um novo split) -- ambos ajudam a controlar overfitting, que é
    # o maior risco aqui dado que cada fold de treino tem só 6 ratos.
    # Combos por passo_pct (SEMSEL): 3*4*4*3*3*2*3*3 = 15552 -- bem mais caro
    # que a grade anterior (~108). Se travar: os primeiros cortes recomendados
    # são tirar "gamma" e reduzir "min_child_weight" pra um valor só.
    PARAM_GRID_XGB_SEMSEL = {
        "clf__n_estimators": [100, 200, 300],          # 3
        "clf__max_depth": [2, 4, 6],                   # 3
        "clf__learning_rate": [0.05, 0.1],             # 2
        "clf__subsample": [0.7, 1.0],                  # 2
        "clf__colsample_bytree": [0.7, 1.0],           # 2
        "clf__min_child_weight": [1, 5],               # 2
        "clf__reg_lambda": [1, 10],                    # 2
        "clf__gamma": [0, 0.5],                        # 2
    }
    PARAM_GRID_XGB_COMSEL = {
        "selector__max_features": TOP_N_FEATURES_GRID,
        **PARAM_GRID_XGB_SEMSEL,
    }

    xgb_base = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=1, tree_method="hist",
    )

    saida_xgb = pipeline_binario_aninhado(
        df_limpo, nome_modelo="XGBoost (50% vs 75%)", estimator=xgb_base,
        param_grid_semsel=PARAM_GRID_XGB_SEMSEL, param_grid_comsel=PARAM_GRID_XGB_COMSEL,
        y_transformer=codificar_y_xgb, inverse_label_fn=decodificar_y_xgb,
    )
else:
    saida_xgb = None
    print("Pulando seção XGBoost -- pacote 'xgboost' não disponível (pip install xgboost).")


## 8. Teste de Permutação (config fixa por fold, simplificação documentada)

Refazer a CV interna para cada uma das permutações multiplicaria o custo já alto
da CV aninhada por `n_permutacoes` -- inviável no prazo. Aqui, para cada fold
externo, reaproveitamos a **mesma configuração** (`passo_pct`, seleção, hiperparâmetros)
já escolhida com os dados reais, e só re-treinamos/reavaliamos com os rótulos
embaralhados **por arquivo** (`stem`) -- preservando a estrutura de que todas as
áreas de um mesmo arquivo têm a mesma condição.

**Limitação a admitir se perguntarem:** a distribuição nula aqui não captura a
variância de "escolher a configuração vencedora", então o p-valor é uma
aproximação prática, não o valor exato de um teste de permutação totalmente
aninhado. Rodamos só para o modelo com maior acurácia observada (mesmo espírito
do notebook original, que testava só o RF).

In [ ]:
def teste_permutacao_aninhado_simplificado(df_long_limpo, resultado_aninhado, estimator,
                                            y_transformer=None, n_permutacoes=50,
                                            random_state=RANDOM_STATE, rotulo="modelo"):
    rng = np.random.RandomState(random_state)
    y_transformer = y_transformer or (lambda y: y)
    df_configs = resultado_aninhado["df_configs_por_fold"].set_index("rato_teste")
    bal_acc_observado = resultado_aninhado["df_resultados"]["balanced_accuracy"].mean()
    ratos = resultado_aninhado["df_resultados"]["rato_teste"].tolist()

    bal_acc_nulo = []
    for p in range(n_permutacoes):
        # embaralha 'condicao' por ARQUIVO (stem), preservando a estrutura de que
        # todas as áreas do mesmo arquivo compartilham a mesma condição
        mapa_stem = df_long_limpo.drop_duplicates("stem")[["stem", "condicao"]].copy()
        mapa_stem["condicao_perm"] = rng.permutation(mapa_stem["condicao"].values)
        df_perm = df_long_limpo.merge(mapa_stem[["stem", "condicao_perm"]], on="stem", how="left")
        df_perm = df_perm.drop(columns=["condicao"]).rename(columns={"condicao_perm": "condicao"})

        bal_accs_fold = []
        for rato_teste in ratos:
            cfg = df_configs.loc[rato_teste]
            df_treino_long = df_perm[df_perm["rato"] != rato_teste]
            df_teste_long = df_perm[df_perm["rato"] == rato_teste]
            X_tr, y_tr, _, X_te, y_te, _, _ = montar_dataset_treino_teste(
                df_treino_long, df_teste_long, cfg["passo_pct"]
            )
            y_tr = y_transformer((y_tr * 100).round().astype(int))
            y_te = y_transformer((y_te * 100).round().astype(int))
            pipe = montar_pipeline_binaria(clone(estimator), cfg["usa_selecao"])
            pipe.set_params(**cfg["params"])
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                bal_accs_fold.append(balanced_accuracy_score(y_te, y_pred))
        bal_acc_nulo.append(np.mean(bal_accs_fold))
        if (p + 1) % 10 == 0:
            print(f"  [{rotulo}] permutação {p+1}/{n_permutacoes}...")

    bal_acc_nulo = np.array(bal_acc_nulo)
    p_valor = (np.sum(bal_acc_nulo >= bal_acc_observado) + 1) / (n_permutacoes + 1)

    print(f"\n[{rotulo}] balanced_accuracy observado: {bal_acc_observado:.3f}")
    print(f"[{rotulo}] balanced_accuracy nulo (média ± dp): {bal_acc_nulo.mean():.3f} ± {bal_acc_nulo.std():.3f}")
    print(f"[{rotulo}] p-valor (permutação, n={n_permutacoes}, config fixa por fold): {p_valor:.4f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(bal_acc_nulo, bins=20, color="#999999", alpha=0.8, label="distribuição nula (rótulos embaralhados por arquivo)")
    ax.axvline(bal_acc_observado, color="crimson", linewidth=2, label=f"observado = {bal_acc_observado:.3f}")
    ax.axvline(0.5, color="black", linestyle=":", label="chance (1/2)")
    ax.set_xlabel("balanced_accuracy (média nested-LOGO)")
    ax.set_ylabel("frequência")
    ax.set_title(f"Teste de permutação (config fixa por fold) -- {rotulo} (n={n_permutacoes}) -- p={p_valor:.4f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return {"bal_acc_observado": bal_acc_observado, "bal_acc_nulo": bal_acc_nulo, "p_valor": p_valor}


saidas_disponiveis = [s for s in [saida_rf, saida_logreg, saida_xgb] if s is not None]
melhor_saida = max(
    saidas_disponiveis,
    key=lambda s: s["resultado"]["df_resultados"]["balanced_accuracy"].mean(),
)
print(f"Modelo com maior balanced_accuracy observada: {melhor_saida['nome_modelo']}")

usar_xgb = XGBOOST_DISPONIVEL and (melhor_saida is saida_xgb)
y_transformer_melhor = codificar_y_xgb if usar_xgb else (lambda y: y)
estimator_melhor = xgb_base if usar_xgb else logreg_base

# Ajuste n_permutacoes conforme o tempo disponível (50 já dá uma ideia; 100-200
# dá um p-valor mais preciso se sobrar tempo).
resultado_permutacao = teste_permutacao_aninhado_simplificado(
    df_limpo, melhor_saida["resultado"], clone(estimator_melhor),
    y_transformer=y_transformer_melhor, n_permutacoes=50, rotulo=melhor_saida["nome_modelo"],
)


## 9. Comparação Final e Exportação dos Resultados

Esta seção existia só no sumário do notebook original -- o código nunca tinha
sido escrito (e a variável `pasta_multiclasse`, usada aqui, nunca havia sido
definida). Ambos os problemas estão corrigidos aqui.

In [ ]:
linhas_comparacao = []
for saida in saidas_disponiveis:
    df_r = saida["resultado"]["df_resultados"]
    linhas_comparacao.append({
        "modelo": saida["nome_modelo"],
        "balanced_accuracy_media": df_r["balanced_accuracy"].mean(),
        "balanced_accuracy_dp": df_r["balanced_accuracy"].std(),
        "f1_macro_media": df_r["f1_macro"].mean(),
        "precision_macro_media": df_r["precision_macro"].mean(),
        "recall_macro_media": df_r["recall_macro"].mean(),
        "n_folds": len(df_r),
    })

df_comparacao = pd.DataFrame(linhas_comparacao).sort_values("balanced_accuracy_media", ascending=False)
print("\n=== Comparação final -- LogReg vs XGBoost (50% vs 75%, CV aninhada) ===")
display(df_comparacao.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_comparacao["modelo"], df_comparacao["balanced_accuracy_media"],
       yerr=df_comparacao["balanced_accuracy_dp"], color="#4C72B0")
ax.axhline(0.5, color="black", linestyle=":", label="chance (1/2)")
ax.set_ylabel("balanced_accuracy (CV aninhada)")
ax.set_title("Comparação final -- 50% vs 75%")
ax.legend()
plt.tight_layout()
plt.show()

df_comparacao.to_csv(pasta_multiclasse / "comparacao_final_logreg_xgb_50_75.csv", index=False)

for saida in saidas_disponiveis:
    nome_arquivo = (
        saida["nome_modelo"].lower()
        .replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct")
    )
    saida["resultado"]["df_resultados"].to_csv(
        pasta_multiclasse / f"resultados_por_fold_{nome_arquivo}.csv", index=False
    )
    saida["resultado"]["df_configs_por_fold"].to_csv(
        pasta_multiclasse / f"configs_por_fold_{nome_arquivo}.csv", index=False
    )

print(f"\n✅ Resultados salvos em: {pasta_multiclasse.resolve()}")
